# Week 84, Day 1: Machine learning concepts

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w084d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import numpy as np

size = np.array([30, 45, 50, 60, 75, 90])           # square metres
rent = np.array([700, 950, 1000, 1150, 1450, 1700])  # pounds a month
slope, intercept = np.polyfit(size, rent, 1)
print(round(slope, 1), round(intercept, 1))
print(round(slope * 65 + intercept))

In [ ]:
import numpy as np

# Six flats: features are size (m²) and rooms; the label is rent.
X = np.array([[30, 1], [45, 2], [50, 2], [60, 3], [75, 3], [90, 4]])
y = np.array([700, 950, 1000, 1150, 1450, 1700])
print(X.shape, y.shape)
print("features of the third flat:", X[2].tolist(), "label:", y[2])

def model(features, weights, bias):
    return features @ weights + bias

print(model(X, np.array([15.0, 50.0]), 200.0).tolist())

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X = np.arange(20).reshape(10, 2)
y = np.arange(10) * 10
order = rng.permutation(len(X))
test, train = order[:3], order[3:]
X_train, X_test, y_train, y_test = X[train], X[test], y[train], y[test]
print(len(X_train), len(X_test))
print(bool((y_train == X_train[:, 0] * 5).all()), "pairs kept together")

labels = np.array(["ham", "ham", "spam", "ham", "ham"])
values, counts = np.unique(labels, return_counts=True)
print("majority baseline:", values[counts.argmax()], counts.max() / len(labels))

In [ ]:
import numpy as np

x = np.array([1.0, 2, 3, 4, 5])
y = np.array([3.1, 4.9, 7.2, 8.8, 11.1])
w = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b = y.mean() - w * x.mean()
print("least squares:", round(w, 3), round(b, 3))

w2, b2, rate = 0.0, 0.0, 0.05
for step in range(2000):
    error = y - (w2 * x + b2)
    w2 -= rate * (-2 * (x * error).mean())
    b2 -= rate * (-2 * error.mean())
print("gradient descent:", round(w2, 3), round(b2, 3))
print("MSE:", round(float(((y - (w * x + b)) ** 2).mean()), 4))

In [ ]:
import numpy as np

rng = np.random.default_rng(4)
x = np.linspace(0, 3, 16)
y = np.sin(x) + rng.normal(0, 0.15, size=16)
train, test = np.arange(16) % 4 != 0, np.arange(16) % 4 == 0
for degree in (1, 3, 9):
    coeffs = np.polyfit(x[train], y[train], degree)
    mse = lambda m: float(((np.polyval(coeffs, x[m]) - y[m]) ** 2).mean())
    print(degree, round(mse(train), 4), round(mse(test), 4))

> **Key idea.** Supervised learning fits a model's parameters to examples with features `X` (a row per example) and labels `y`: regression for numbers, classification for categories; clustering groups unlabelled examples. Shuffle features and labels together and split off a test set before training; judge models on it, and only against a baseline (the training mean, or the majority class). Losses measure error: MSE punishes large errors, MAE is in the label's units, accuracy is the fraction right. Least squares fits a line exactly; gradient descent repeatedly steps parameters against the gradient, scaled by a learning rate that mustn't be too large. Too simple underfits (bad everywhere); too flexible overfits (training error far below test error). k-nearest neighbours classifies by the majority of the closest `k` training points; `k` sets its flexibility. Models learn what's in their data, flaws included.

### Exercise: An honest split

Write `split(X, y, test_share, rng)`: shuffle the row numbers with `rng.permutation(len(X))`, put the first `round(len(X) × test_share)` of them in the test set and the rest in the training set, and return `(X_train, X_test, y_train, y_test)`, keeping each example's features and label together.

In [ ]:
import numpy as np

def split(X, y, test_share, rng):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_split():\n    \"pairs together, sizes right, reproducible\"\n    X = np.arange(40).reshape(20, 2)\n    y = X[:, 0] * 3\n    X_train, X_test, y_train, y_test = split(X, y, 0.25, np.random.default_rng(1))\n    assert len(X_test) == 5 and len(X_train) == 15 and len(y_test) == 5\n    assert (y_train == X_train[:, 0] * 3).all() and (y_test == X_test[:, 0] * 3).all(), \"keep pairs together\"\n    assert sorted(np.concatenate([y_train, y_test]).tolist()) == sorted(y.tolist()), \"every example used once\"\n    again = split(X, y, 0.25, np.random.default_rng(1))\n    assert (again[1] == X_test).all(), \"the same seed gives the same split\"\n    order = np.random.default_rng(1).permutation(20)\n    assert (X_test == X[order[:5]]).all(), \"use rng.permutation as described\"")

### Exercise: Scores and baselines

Write `mse(y, pred)`, `mae(y, pred)` and `accuracy(y, pred)`, each returning a float rounded to 3 places. Then `mean_baseline(y_train, n)`: an array of `n` copies of the training mean, and `majority_baseline(y_train, n)`: an array of `n` copies of the most common training label (the smallest such label if there's a tie).

In [ ]:
import numpy as np

def mse(y, pred):
    ___

def mae(y, pred):
    ___

def accuracy(y, pred):
    ___

def mean_baseline(y_train, n):
    ___

def majority_baseline(y_train, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_metrics():\n    \"errors and baselines\"\n    y, p = np.array([3.0, 5.0, 8.0]), np.array([2.0, 5.0, 11.0])\n    assert mse(y, p) == 3.333 and mae(y, p) == 1.333\n    assert accuracy(np.array([\"a\", \"b\", \"a\", \"a\"]), np.array([\"a\", \"a\", \"a\", \"b\"])) == 0.5\n    assert mean_baseline(np.array([10, 20, 60]), 2).tolist() == [30.0, 30.0]\n    assert majority_baseline(np.array([\"ham\", \"spam\", \"ham\"]), 3).tolist() == [\"ham\", \"ham\", \"ham\"]\n    assert majority_baseline(np.array([2, 1, 2, 1]), 1).tolist() == [1]")

### Exercise: Least squares

Write `fit_line(x, y)`: the least-squares slope and intercept, from the formulas (not `np.polyfit`), as plain floats. `predict(x, slope, intercept)`: the predictions. And `r2(y, pred)`: the **coefficient of determination**, `1 − Σ(y − ŷ)² / Σ(y − ȳ)²` (1 is perfect; 0 is no better than predicting the mean), rounded to 3 places.

In [ ]:
import numpy as np

def fit_line(x, y):
    ___

def predict(x, slope, intercept):
    ___

def r2(y, pred):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_line():\n    \"the formulas\"\n    size = np.array([30, 45, 50, 60, 75, 90])\n    rent = np.array([700, 950, 1000, 1150, 1450, 1700])\n    w, b = fit_line(size, rent)\n    pw, pb = np.polyfit(size, rent, 1)\n    assert abs(w - pw) < 1e-9 and abs(b - pb) < 1e-6 and isinstance(w, float)\n    assert r2(rent, predict(size, w, b)) == 0.996\n    assert r2(rent, np.full(6, rent.mean())) == 0.0\n    assert \"polyfit\" not in source, \"use the formulas\"")

### Exercise: Walking downhill

Write `descend(x, y, rate, steps)`: fit `ŷ = w·x + b` by gradient descent on the MSE, starting from `w = b = 0`. At each step, compute the predictions and the gradients `dw = −2·mean(x·(y − ŷ))` and `db = −2·mean(y − ŷ)`, record the MSE **before** updating, then update both parameters. Return `(w, b, losses)`, with `losses` the list of MSEs (one per step). Also write `diverges(losses)`: `True` if the last loss is larger than the first (the learning rate was too big).

In [ ]:
import numpy as np

def descend(x, y, rate, steps):
    ___

def diverges(losses):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_descent():\n    \"converges to least squares, or blows up\"\n    x = np.array([1.0, 2, 3, 4, 5])\n    y = np.array([3.1, 4.9, 7.2, 8.8, 11.1])\n    w, b, losses = descend(x, y, 0.05, 2000)\n    assert abs(w - 1.99) < 0.01 and abs(b - 1.05) < 0.01, (w, b)\n    assert len(losses) == 2000 and losses[0] > losses[-1] and abs(losses[0] - float((y ** 2).mean())) < 1e-9\n    assert all(a >= c - 1e-9 for a, c in zip(losses, losses[1:])), \"the loss falls every step\"\n    assert not diverges(losses)\n    w, b, losses = descend(x, y, 0.2, 30)\n    assert diverges(losses), \"too large a rate\"")

### Exercise: k-nearest neighbours

Write `knn_predict(X_train, y_train, X_new, k)`: for each row of `X_new`, find the `k` training rows nearest to it (straight-line distance; compute all distances at once with broadcasting), and predict the most common label among them. If labels tie, choose the tied label of the **nearest** of those neighbours. Return an array of predictions.

In [ ]:
import numpy as np
from collections import Counter

def knn_predict(X_train, y_train, X_new, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_knn():\n    \"the closest k vote\"\n    X = np.array([[1, 1], [1, 2], [2, 1], [6, 6], [6, 7], [7, 6]], dtype=float)\n    y = np.array([\"red\", \"red\", \"red\", \"blue\", \"blue\", \"blue\"])\n    new = np.array([[1.5, 1.5], [6.5, 6.2], [4, 4]], dtype=float)\n    assert knn_predict(X, y, new, 3).tolist() == [\"red\", \"blue\", \"red\"]\n    assert knn_predict(X, y, X, 1).tolist() == y.tolist(), \"k=1 reproduces the training labels\"\n    assert knn_predict(X, y, np.array([[3.0, 3.0]]), 6).tolist() == [\"red\"], \"a 3-3 tie goes to the nearest's label\"\n    assert knn_predict(X, y, np.array([[4.6, 4.6]]), 6).tolist() == [\"blue\"]")

### Exercise: Finding the right flexibility

Write `poly_errors(x_train, y_train, x_test, y_test, degrees)`: for each degree, fit a polynomial to the training data with `np.polyfit` and return a dict from degree to `(train_mse, test_mse)`, each rounded to 4 places. Then `best_degree(errors)`: the degree with the lowest **test** error, and `overfitting(errors)`: the sorted degrees whose test error is more than twice their training error.

In [ ]:
import numpy as np

def poly_errors(x_train, y_train, x_test, y_test, degrees):
    ___

def best_degree(errors):
    ___

def overfitting(errors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_fit():\n    \"train error falls; test error doesn't\"\n    x = np.linspace(0, 3, 16)\n    noise = np.array([0.1, -0.12, 0.05, 0.14, -0.08, 0.02, -0.15, 0.09, 0.11, -0.04, 0.07, -0.1, 0.03, 0.12, -0.06, 0.0])\n    y = np.sin(x) + noise\n    test = np.arange(16) % 4 == 1\n    errors = poly_errors(x[~test], y[~test], x[test], y[test], [1, 2, 3, 6])\n    assert set(errors) == {1, 2, 3, 6}\n    trains = [errors[d][0] for d in (1, 2, 3, 6)]\n    assert trains == sorted(trains, reverse=True), \"more flexible, lower training error\"\n    assert errors[1][0] > 0.05 and errors[1][1] > 0.05, \"a straight line underfits a curve\"\n    assert best_degree(errors) in (2, 3) and 6 in overfitting(errors) and 1 not in overfitting(errors)")

### Bug hunt: A perfect score that means nothing

One-nearest-neighbour is scored on the same points it was trained on, so every point finds itself: `accuracy: 1.0`. Score it on the test points instead, so it prints `accuracy: 0.75`.

In [ ]:
import numpy as np

X_train = np.array([[1.0], [2.0], [3.0], [6.0], [7.0], [8.0]])
y_train = np.array([0, 0, 1, 1, 1, 0])
X_test = np.array([[1.5], [2.6], [6.4], [7.6]])
y_test = np.array([0, 1, 1, 1])

def nearest_label(points):
    d = np.abs(points[:, None, 0] - X_train[None, :, 0])
    return y_train[d.argmin(axis=1)]

print("accuracy:", float((nearest_label(X_train) == y_train).mean()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unseen data\"\n    expect_output(\"accuracy: 0.75\")")

### Bug hunt: Uphill all the way

Gradient descent should walk **against** the gradient, but this steps along it, so the slope runs off to a huge number. Fix the update so it prints `w = 2.0`.

In [ ]:
import numpy as np

x = np.array([1.0, 2.0, 3.0])
y = np.array([2.0, 4.0, 6.0])
w = 0.0
for _ in range(200):
    grad = -2 * (x * (y - w * x)).mean()
    w += 0.05 * grad
print("w =", round(w, 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"downhill\"\n    expect_output(\"w = 2.0\")")

### Bug hunt: Shuffled apart

Features and labels are shuffled with **different** permutations, so each label now belongs to the wrong example: it prints `pairs intact: False`. Shuffle both with one permutation so it prints `pairs intact: True`.

In [ ]:
import numpy as np

rng = np.random.default_rng(5)
X = np.arange(10)
y = X * 2
X_shuffled = X[rng.permutation(10)]
y_shuffled = y[rng.permutation(10)]
print("pairs intact:", bool((y_shuffled == X_shuffled * 2).all()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one permutation\"\n    expect_output(\"pairs intact: True\")")

### Puzzle: An honest experiment

Arrange the steps of a fair comparison: split first, fit on the training set only, then compare the model with a baseline on the test set. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # "regression", "classification" or "clustering": predicting tomorrow's rainfall in mm
q = ___   # the same, for grouping customers by what they buy, with no labels
r = ___   # the accuracy of always predicting "no" when 92 of 100 test labels are "no"
s = ___   # "underfitting" or "overfitting": training error 0.01, test error 0.40
t = ___   # the MSE of predictions [2, 4] for labels [1, 7]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"regression\" and q == \"clustering\" and r == 0.92, \"p, q and r\"\n    assert s == \"overfitting\" and t == 5.0, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: k-means clustering

Clustering needs no labels. **k-means** groups points into `k` clusters: start with `k` centres, then repeat two steps until nothing changes: **assign** each point to its nearest centre, and **update** each centre to the mean of its points. Write `kmeans(X, k, max_steps=100)`: start with the first `k` rows as the centres; a point equally near two centres joins the one with the lower number; a centre with no points stays where it is. Stop when an assignment step changes nothing (or after `max_steps` assignment steps). Return a dict with `centres` (an array, rounded to 3 places), `labels` (each point's cluster number, as a list), `steps` (the number of assignment steps run, including the last one that changed nothing) and `inertia` (the sum of squared distances from each point to its centre, rounded to 3 places). Then `best_k(X, ks)`: a dict from each `k` to its inertia, which always falls as `k` grows (look for the "elbow" where it stops falling fast).

In [ ]:
import numpy as np

def kmeans(X, k, max_steps=100):
    ___

def best_k(X, ks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\nX = np.array([[1.0, 1.0], [8.0, 8.0], [1.5, 2.0], [8.5, 7.5], [1.0, 1.5], [9.0, 8.0], [5.0, 1.0], [5.5, 1.5]])\n\ndef test_kmeans():\n    \"assign, update, repeat\"\n    r = kmeans(X, 2)\n    assert r[\"labels\"] == [0, 1, 0, 1, 0, 1, 0, 0]\n    assert r[\"centres\"].tolist() == [[2.8, 1.4], [8.5, 7.833]]\n    assert r[\"steps\"] == 2 and r[\"inertia\"] == 21.667\n    r3 = kmeans(X, 3)\n    assert r3[\"labels\"] == [0, 1, 0, 1, 0, 1, 2, 2]\n    assert r3[\"centres\"].tolist() == [[1.167, 1.5], [8.5, 7.833], [5.25, 1.25]]\n    assert r3[\"inertia\"] < r[\"inertia\"]\n\ndef test_edges():\n    \"limits and the elbow\"\n    one = kmeans(X, 2, max_steps=1)\n    assert one[\"steps\"] == 1\n    inertias = best_k(X, [1, 2, 3, 4])\n    assert list(inertias) == [1, 2, 3, 4]\n    values = list(inertias.values())\n    assert values == sorted(values, reverse=True) and inertias[1] == round(float(((X - X.mean(axis=0)) ** 2).sum()), 3)")

## Warm-up

Nobody told the program what a square metre is worth: it **learned** a slope and an intercept from six examples, and used them to predict the rent of a flat it has never seen. That's machine learning in miniature: examples in, **parameters** out, predictions for new cases.

## Learning from examples

Each example has **features** (the inputs: size, number of rooms, distance to a station) and, for the problems this week, a **label** (the answer: the rent). Arranged as data, the features are a table `X` with one row per example and one column per feature, and the labels a column `y`. A **model** is a function from features to a prediction, with **parameters** (a slope and intercept, or millions of weights) that **training** chooses to make its predictions on the examples good.

Three kinds of problem cover most of what you'll meet:

- **Regression**: the label is a number (rent, temperature, journey time).
- **Classification**: the label is a category (spam or not, which digit, which species).
- **Clustering**: there are no labels; the program groups similar examples (customers who shop alike), and people decide what the groups mean.

Problems with labels are **supervised** learning; clustering is **unsupervised**. (A fourth kind, reinforcement learning, learns from rewards for actions, as in game-playing programs.)

A model learns whatever is in its examples, including their mistakes and their unfairness: a model trained on past decisions copies the patterns in those decisions. Where predictions affect people, what the data represents matters as much as the code.

`features @ weights + bias` is a linear model with two weights: a guess at the parameters gives predictions; training's job is to find better ones.

## Training and testing

A model can do well on its training examples just by **remembering** them. What matters is how it does on examples it hasn't seen, so before training, set some aside: shuffle the rows and split them into a **training set** (often 70 to 80%) for choosing parameters and a **test set** for measuring, used only at the end. Shuffle features and labels **together** (one permutation of row numbers for both), or the pairs are broken.

Then compare every model with a **baseline**, the score you'd get without learning anything: predict the training labels' **mean** for regression, or the **most common class** for classification. If 90% of emails aren't spam, a "model" that always says "not spam" is 90% accurate; a real model has to beat that to be worth anything.

## Loss, and fitting a line

To choose parameters, training needs a number saying how wrong the predictions are: a **loss**. For regression, the **mean squared error** `MSE = mean((y − ŷ)²)` punishes big mistakes heavily; the **mean absolute error** `MAE = mean(|y − ŷ|)` is in the label's own units and easier to explain. For classification, **accuracy** is the fraction predicted correctly.

For a straight line `ŷ = w·x + b`, the parameters with the smallest MSE have a formula (**least squares**): `w = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)²` and `b = ȳ − w·x̄`. Most models have no formula, so they use **gradient descent**: start with any parameters, work out which direction makes the loss smaller (the **gradient**, the slope of the loss), take a small step that way, and repeat. For the line, the gradients of the MSE are `∂/∂w = −2·mean(x·(y − ŷ))` and `∂/∂b = −2·mean(y − ŷ)`; each step does `w -= rate × ∂/∂w`. The **learning rate** decides the step size: too small is slow, too large overshoots and diverges.

Both reach the same line: gradient descent found by walking downhill what the formula computes directly. Every neural network is trained the same way, with more parameters.

## Too simple, too flexible

A model that's too simple **underfits**: a straight line through data that curves is wrong everywhere, on training and test data alike. A model that's too flexible **overfits**: a curve wiggly enough to pass through every training point has learned the noise as well as the pattern, and it does badly on new points. The sign of overfitting is a **training error much lower than the test error**.

**k-nearest neighbours** shows the balance clearly. To classify a new point, find the `k` training points closest to it and take the most common label among them. With `k = 1`, every training point is classified by itself (perfect on training data, jumpy on new data); a large `k` smooths more, and at `k = n` it predicts the majority class everywhere. Choosing `k`, or a polynomial's degree, is choosing flexibility, and tomorrow's tools choose it by testing.

Degree 1 is poor on both sets; degree 9 is nearly perfect on training points and worse on the test points it never saw.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a decision you make often (whether to take an umbrella, which route to work) and write down the features you'd give a model and the label it would predict. Is it regression or classification? What would the baseline be, and how would you collect honest test examples? Tomorrow is **scikit-learn**, which does today's work (and much more) in a few lines, with the same ideas underneath.